# Custom Environment 2

In [ ]:
import math
import random
import time
from typing import Tuple, List

import gymnasium
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt



from stable_baselines3.common.vec_env import DummyVecEnv


In [1]:
from enum import Enum
import gymnasium as gym
from gymnasium import spaces
import pygame
import numpy as np
import math
import random
from typing import Tuple, List, Dict


class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 8}

    def __init__(self,
                 grid_size: Tuple[int,int] = (64,64),
                 v_max: float = 0.6,
                 omega_max: float = 2.0,
                 max_steps: int = 500,
                 render_mode: str = None,
                 seed: int = None,
                 num_moving_obstacles: int = 3,
                 moving_obstacle_speed: float = 0.15):
        super().__init__()
        self.grid_w, self.grid_h = grid_size
        self.v_max = v_max
        self.omega_max = omega_max
        self.max_steps = max_steps
        self.render_mode = render_mode
        self.window_size = 512

        self.num_moving_obstacles = num_moving_obstacles
        self.moving_obstacle_speed = moving_obstacle_speed

        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, -math.pi], dtype=np.float32),
            high=np.array([1.0, 1.0, math.pi], dtype=np.float32),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=np.array([0.0, -self.omega_max], dtype=np.float32),
            high=np.array([self.v_max, self.omega_max], dtype=np.float32),
            dtype=np.float32
        )

        # occupancy grid of static obstacles (0 free, 1 occupied)
        self.occupancy = np.zeros((self.grid_h, self.grid_w), dtype=np.uint8)
        self.obstacles: List[Tuple[int,int,int,int]] = []
        self._build_default_map()

        # moving obstacles
        self.moving_obstacles: List[Dict] = []
        self._init_moving_obstacles()

        self.window = None
        self.clock = None

        self.seed(seed)

    def seed(self, s=None):
        self.np_random, seed = gym.utils.seeding.np_random(s)
        random.seed(int(seed % (2**32 - 1)))
        np.random.seed(int(seed % (2**32 - 1)))
        return [int(seed % (2**32 - 1))]

    def _build_default_map(self):
        self.grid_w, self.grid_h = 8, 8
        self.occupancy = np.zeros((self.grid_h, self.grid_w), dtype=np.uint8)
        self.obstacles = []
    
        for y in range(self.grid_h):
            for x in range(self.grid_w):
                if self.occupancy[y, x] == 1:
                    self._add_obstacle(x, y, x+1, y+1)
    
        self._reserved_start = (0, 7)
        self._reserved_goal = (5, 3)

    def _init_moving_obstacles(self):
        self.moving_obstacles = []
        free_cells = np.argwhere(self.occupancy == 0)
        if len(free_cells) == 0:
            return

        k = min(self.num_moving_obstacles, len(free_cells))
        chosen_indices = self.np_random.choice(len(free_cells), size=k, replace=False)
        cell_radius = 0.2 / max(self.grid_w, self.grid_h)

        for idx in chosen_indices:
            gy, gx = free_cells[idx]
            pos = np.array(self._grid_to_world(gx, gy), dtype=np.float32)
            angle = self.np_random.uniform(-math.pi, math.pi)
            speed = float(self.np_random.uniform(0.05, self.moving_obstacle_speed))
            vel = np.array([math.cos(angle) * speed, math.sin(angle) * speed], dtype=np.float32)
            self.moving_obstacles.append({'pos': pos.copy(), 'vel': vel.copy(), 'radius': cell_radius})

    def _add_obstacle(self, x1, y1, x2, y2):
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(self.grid_w, x2), min(self.grid_h, y2)
        if x2 > x1 and y2 > y1:
            self.occupancy[y1:y2, x1:x2] = 1
            self.obstacles.append((x1, y1, x2, y2))

    def _grid_to_world(self, gx, gy) -> Tuple[float,float]:
        return ((gx + 0.5) / self.grid_w, (gy + 0.5) / self.grid_h)

    def _world_to_grid(self, x, y) -> Tuple[int,int]:
        gx = min(self.grid_w-1, max(0, int(math.floor(x * self.grid_w))))
        gy = min(self.grid_h-1, max(0, int(math.floor(y * self.grid_h))))
        return gx, gy

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._init_moving_obstacles()

        sx, sy = self._grid_to_world(*self._reserved_start)
        gx, gy = self._grid_to_world(*self._reserved_goal)

        self.state = np.array([sx, sy], dtype=np.float32)
        self.theta = random.uniform(-math.pi, math.pi)
        self.goal = np.array([gx, gy], dtype=np.float32)
        self.goal_radius = 0.08
        self.step_count = 0
        self.last_action = np.zeros(2, dtype=np.float32)

        obs = np.array([self.state[0], self.state[1], self.theta], dtype=np.float32)
        info = {"distance": float(np.linalg.norm(self.state - self.goal))}
        if self.render_mode == "human":
            self._render_frame()
        return obs, info

    def _update_moving_obstacles(self, dt: float):
        for obs in self.moving_obstacles:
            obs['pos'] = obs['pos'] + obs['vel'] * dt
            for i in (0,1):
                if obs['pos'][i] < 0.0:
                    obs['pos'][i] = -obs['pos'][i]
                    obs['vel'][i] = -obs['vel'][i]
                elif obs['pos'][i] > 1.0:
                    obs['pos'][i] = 2.0 - obs['pos'][i]
                    obs['vel'][i] = -obs['vel'][i]

    @staticmethod
    def _wrap_to_pi(angle: float) -> float:
        return (angle + math.pi) % (2*math.pi) - math.pi

    def _compute_obstacle_penalty(self, pos: np.ndarray, radius: float = 0.02) -> float:
        """
        Simple proximity penalty to static obstacles: higher when near occupied cells.
        """
        gy, gx = self._world_to_grid(pos[0], pos[1])
        penalty = 0.0
        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                ny, nx = gy + dy, gx + dx
                if 0 <= ny < self.grid_h and 0 <= nx < self.grid_w and self.occupancy[ny, nx] == 1:
                    cell_center = np.array(self._grid_to_world(nx, ny))
                    dist = np.linalg.norm(cell_center - pos)
                    penalty += max(0.0, (radius + 0.08 - dist))
        return penalty

    def _check_collision_with_moving(self, pos: np.ndarray) -> bool:
        ego_radius = 0.02
        for mob in self.moving_obstacles:
            if np.linalg.norm(mob['pos'] - pos) <= (ego_radius + mob['radius']):
                return True
        return False

    # (VO penalty computation stays same as your version — omitted here for brevity)
    # ...

    def step(self, action: np.ndarray):
        action = np.clip(action, self.action_space.low, self.action_space.high).astype(np.float32)
        v_cmd, omega_cmd = float(action[0]), float(action[1])

        dt = 0.3
        self._update_moving_obstacles(dt)

        self.theta += omega_cmd * dt
        dx = v_cmd * math.cos(self.theta) * dt
        dy = v_cmd * math.sin(self.theta) * dt

        prev_dist = np.linalg.norm(self.state - self.goal)
        new_pos = np.clip(self.state + np.array([dx, dy]), 0.0, 1.0)

        gx, gy = self._world_to_grid(new_pos[0], new_pos[1])
        collision_static = bool(self.occupancy[gy, gx] == 1)
        collision_moving = self._check_collision_with_moving(new_pos)
        collision = collision_static or collision_moving

        self.state = new_pos
        new_dist = np.linalg.norm(self.state - self.goal)
        progress = (prev_dist - new_dist) * 50.0

        ego_vel_vector = np.array([v_cmd * math.cos(self.theta), v_cmd * math.sin(self.theta)], dtype=np.float32)
        vo_penalty = self._compute_vo_penalty(self.state, ego_vel_vector)
        obstacle_penalty = -1.5 * self._compute_obstacle_penalty(self.state)
        smooth_penalty = -0.1 * np.linalg.norm(action - self.last_action)
        time_pen = -0.01

        reward = progress + obstacle_penalty + smooth_penalty + time_pen - float(vo_penalty)

        self.last_action = np.copy(action)
        self.step_count += 1

        terminated = truncated = False
        info = {}

        if collision:
            terminated = True
            reward = -10.0
            info = {"reason": "collision"}
        elif new_dist <= self.goal_radius:
            terminated = True
            reward += 100.0
            info = {"reason": "goal"}
            print("Goal reached!")
        elif self.step_count >= self.max_steps:
            truncated = True
            info = {"reason": "timeout"}

        obs = np.array([self.state[0], self.state[1], self.theta], dtype=np.float32)
        return obs, float(reward), terminated, truncated, info

    def render(self):
        if self.render_mode == "rgb_array":
            return self._render_frame()
        elif self.render_mode == "human":
            self._render_frame()
        else:
            return None

    def _render_frame(self):
        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_w = self.window_size / self.grid_w
        pix_h = self.window_size / self.grid_h

        obs_coords = np.argwhere(self.occupancy == 1)
        for (gy, gx) in obs_coords:
            rect = pygame.Rect(int(gx * pix_w),
                               int(self.window_size - (gy + 1) * pix_h),
                               int(pix_w) + 1, int(pix_h) + 1)
            pygame.draw.rect(canvas, (0, 0, 0), rect)

        gx_f = int(self.goal[0] * self.window_size)
        gy_f = int(self.window_size - self.goal[1] * self.window_size)
        goal_size = int(min(pix_w, pix_h) * 0.8)
        pygame.draw.rect(canvas, (255, 0, 0),
                         pygame.Rect(gx_f - goal_size // 2, gy_f - goal_size // 2, goal_size, goal_size))

        for mob in self.moving_obstacles:
            ox = int(mob['pos'][0] * self.window_size)
            oy = int(self.window_size - mob['pos'][1] * self.window_size)
            rpx = max(2, int(mob['radius'] * self.window_size * 2))
            pygame.draw.circle(canvas, (200, 0, 0), (ox, oy), rpx)

        ax = int(self.state[0] * self.window_size)
        ay = int(self.window_size - self.state[1] * self.window_size)
        pygame.draw.circle(canvas, (0, 0, 255), (ax, ay), int(min(pix_w, pix_h) * 0.4))

        if self.render_mode == "human":
            self.window.blit(canvas, canvas.get_rect())
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])
        else:
            arr = pygame.surfarray.array3d(canvas)
            return np.transpose(arr, (1, 0, 2))
        
    def _compute_obstacle_penalty(self, pos: np.ndarray) -> float:
        """
        Compute a small penalty based on proximity to static obstacles.
        """
        gx, gy = self._world_to_grid(pos[0], pos[1])
        penalty = 0.0
        max_range = 2  # look at 2 grid cells around agent
        for dy in range(-max_range, max_range + 1):
            for dx in range(-max_range, max_range + 1):
                nx, ny = gx + dx, gy + dy
                if 0 <= nx < self.grid_w and 0 <= ny < self.grid_h:
                    if self.occupancy[ny, nx] == 1:
                        dist = math.hypot(dx, dy)
                        if dist < 1e-6:
                            penalty += 1.0
                        else:
                            penalty += 1.0 / (dist ** 2)
        return float(penalty)
    

    def _compute_vo_penalty(self, ego_pos: np.ndarray, ego_vel: np.ndarray,
                            vo_weight: float = 2.0, ttc_weight: float = 1.0, ttc_tau: float = 0.5,
                            safe_clearance: float = 0.02) -> float:
        """
        Compute Velocity Obstacle penalty considering both static and moving obstacles.
        Returns a positive penalty (higher = more dangerous).
        """
        total_pen = 0.0
        ego_radius = 0.02

        # Prepare list of obstacles (static + moving)
        obs_list = []
        cell_radius = 0.5 / max(self.grid_w, self.grid_h)
        obs_cells = np.argwhere(self.occupancy == 1)
        for (gy, gx) in obs_cells:
            obs_pos = np.array(self._grid_to_world(gx, gy), dtype=np.float32)
            obs_list.append({'pos': obs_pos, 'vel': np.array([0.0, 0.0]), 'radius': cell_radius})
        obs_list.extend(self.moving_obstacles)

        for obs in obs_list:
            p = obs['pos'] - ego_pos
            dist = np.linalg.norm(p)
            if dist < 1e-6:
                total_pen += 10.0 * vo_weight
                continue

            R = ego_radius + obs['radius']
            v_rel = obs['vel'] - ego_vel
            v_rel_norm = np.linalg.norm(v_rel) + 1e-8

            # VO cone half-angle
            theta = math.asin(min(1.0, R / dist))
            alpha = math.atan2(p[1], p[0])
            beta = math.atan2(v_rel[1], v_rel[0])
            phi = abs((beta - alpha + math.pi) % (2 * math.pi) - math.pi)

            approaching = np.dot(p, v_rel) < 0
            inside_cone = phi < theta and approaching

            if inside_cone:
                angular_pen = (theta - phi) / theta
                ttc = -np.dot(p, v_rel) / (np.dot(v_rel, v_rel) + 1e-8)
                ttc_pen = ttc_weight * math.exp(-ttc / (ttc_tau + 1e-8))
                total_pen += vo_weight * angular_pen + ttc_pen

            # extra clearance penalty
            if dist < (R + safe_clearance):
                total_pen += vo_weight * (R + safe_clearance - dist) / (R + safe_clearance)

        return float(total_pen)


    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()
            self.window = None
            self.clock = None


### PPO

In [2]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_checker import check_env
import numpy as np

# --- check environment once ---
check_env(GridWorldEnv(render_mode=None), warn=True)

# --- create training environment ---
def make_train_env():
    return GridWorldEnv(render_mode=None)

train_env = DummyVecEnv([make_train_env])

# --- define PPO model ---
model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=1e-4,       # smaller LR → more stable
    n_steps=4096,             # longer rollouts
    batch_size=256,           # smoother gradient updates
    n_epochs=10,
    gamma=0.98,
    clip_range=0.2,
)

# --- train ---
print("🚀 Starting PPO training...")
model.learn(total_timesteps=200000)   # can increase for better performance
print("✅ Training finished. Saving model...")
model.save("ppo_velocity_obstacle")

# --- evaluation ---
eval_env = GridWorldEnv(render_mode="human")
obs, info = eval_env.reset()

print("🎯 Evaluating trained PPO agent...")
for step in range(300):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    eval_env.render()
    if terminated or truncated:
        obs, info = eval_env.reset()

eval_env.close()
train_env.close()


c:\Users\snigd\miniconda3\envs\myenv\lib\site-packages\stable_baselines3\common\env_checker.py:462: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(


Using cpu device
🚀 Starting PPO training...
-----------------------------
| time/              |      |
|    fps             | 1119 |
|    iterations      | 1    |
|    time_elapsed    | 3    |
|    total_timesteps | 4096 |
-----------------------------
Goal reached!
Goal reached!
------------------------------------------
| time/                   |              |
|    fps                  | 851          |
|    iterations           | 2            |
|    time_elapsed         | 9            |
|    total_timesteps      | 8192         |
| train/                  |              |
|    approx_kl            | 0.0025433972 |
|    clip_fraction        | 0.0149       |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.84        |
|    explained_variance   | -0.00879     |
|    learning_rate        | 0.0001       |
|    loss                 | 13           |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00319     |
|    std                  | 0.

### 1st paper
Concentrates on stability even when the goal is to maximize rewards.
A robot arm might learn a policy that gives high reward but overshoots and oscillates.A drone might crash during exploration.Classical control (PID, LQR, etc.) guarantees stability, but RL usually doesn’t.

Concentrates on the question that:
Can we make RL learn a policy that’s both optimal and stable?

So there is a Lyapunov function V(x) which is like an energy function for our system. Its always positive (at equi it is 0) and it decreases over time as the system moves toward the stable point. 
V(x) is like a "certificate" that your system won’t blow up.

So there is a policy network which decides the control action and a lyapunov network which predicts the stability function V(x).

These papers combine RL + Lyapunov stability theory using neural networks.
Instead of designing a Lyapunov function by hand, they let a neural network learn V(x) that satisfies stability conditions. 

And they jointly train both to make sure V(xt+1)-V(xt)<=0 
This ensures that as the agent acts, the system's energy always decreases which guarantees stability.

### In the 2nd paper methodology
the second paper does relate to markov processes in the context of reinforcement learning particularly CMDPs. The paper discusses using Lyapunov stability theory to address stability and safety in RL.

MDPs :Markov decision processes are useful in systems where there are a variety of choices available in uncertain environments. A crucial aspect of Markovian decision processes is that the transition probabilities and outcomes depend only on the current state of the agent and system, not any past states. However, in real life, past states and actions often significantly impact our future decisions and their outcomes.
